# Model training and optimization

# Day 3 - Explore delay propagation and establish baselines

This notebook analyzes linked Southwest (WN) aircraft-flight pairs from January-April 2026. The target is whether the current flight departs at least 15 minutes late (`DEP_DEL15`). It explores delay patterns by hour, weekday, origin airport, previous-flight arrival delay, and scheduled turnaround slack, then compares a majority-class baseline with an initial logistic-regression model.

**Evaluation design:** January-February are used for the initial baseline, January for candidate fitting, and February for model/threshold selection. March is a previously inspected comparison. The selected model is then refit on January-March and evaluated on April as the fresh final holdout. All imputing, scaling, and encoding are fitted on training data only.

**Interpretation limit:** this is an initial historical benchmark, not an operational dispatch recommendation. The available sample covers four months and one carrier.


In [1]:

from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Find the workspace root from the notebook working directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "processed").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data" / "processed").is_dir():
    raise FileNotFoundError("Open this notebook from the flight project folder.")
PAIR_PATH = ROOT / "data" / "processed" / "wn_aircraft_pairs_2026-01_to_2026-04.csv"
REPORT_DIR = ROOT / "reports" / "metrics"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)



## 1. Load the linked-pair data and inspect the target

Each row represents a current flight and the immediately preceding flight assigned to the same aircraft. The pair construction rules and raw-data quality profile are documented in the Day 2 notebook. Current-flight actual delay is used to create the target, but will not be included among the predictors.


In [2]:

if not PAIR_PATH.is_file():
    raise FileNotFoundError(f"Run notebooks/02_data_preprocessing.ipynb first; missing {PAIR_PATH}")

pairs = pd.read_csv(PAIR_PATH, low_memory=False, parse_dates=["FLIGHT_DATE"])
required = {
    "FLIGHT_DATE", "DEP_DEL15", "DEP_DELAY_MIN", "PREV_ARR_DELAY_MIN",
    "SCHEDULED_TURNAROUND_SLACK_MIN", "ORIGIN", "DEST",
    "DEPARTURE_HOUR_LOCAL", "DAY_OF_WEEK", "MONTH",
}
missing = sorted(required - set(pairs.columns))
if missing:
    raise ValueError(f"Pair file is missing required fields: {missing}")
pairs["FLIGHT_DATE"] = pd.to_datetime(pairs["FLIGHT_DATE"], errors="coerce")
pairs = pairs.dropna(subset=["FLIGHT_DATE", "DEP_DEL15"]).copy()
pairs["DEP_DEL15"] = pairs["DEP_DEL15"].astype(int)

print("Pair table shape:", pairs.shape)
print("Date range:", pairs["FLIGHT_DATE"].min().date(), "to", pairs["FLIGHT_DATE"].max().date())
print("Target counts:")
print(pairs["DEP_DEL15"].value_counts().rename(index={0: "under 15 min", 1: "15+ min"}))
print("Overall 15+ minute departure-delay rate:", round(pairs["DEP_DEL15"].mean(), 4))
print("Missing values in candidate fields:")
print(pairs[list(required)].isna().sum().sort_values(ascending=False).to_string())


Pair table shape: (423660, 22)
Date range: 2026-01-01 to 2026-04-30
Target counts:
DEP_DEL15
under 15 min    321558
15+ min         102102
Name: count, dtype: int64
Overall 15+ minute departure-delay rate: 0.241
Missing values in candidate fields:
DEST                              0
DEPARTURE_HOUR_LOCAL              0
FLIGHT_DATE                       0
DEP_DELAY_MIN                     0
ORIGIN                            0
SCHEDULED_TURNAROUND_SLACK_MIN    0
DEP_DEL15                         0
DAY_OF_WEEK                       0
MONTH                             0
PREV_ARR_DELAY_MIN                0


In [ ]:
# Recreate the chronological split and baseline feature definitions used by the original modeling workflow.
cutoff = pd.Timestamp("2026-03-01")
target = "DEP_DEL15"
numeric_features = ["PREV_ARR_DELAY_MIN", "SCHEDULED_TURNAROUND_SLACK_MIN"]
categorical_features = ["ORIGIN", "DEST", "DEPARTURE_HOUR_LOCAL", "DAY_OF_WEEK"]
feature_columns = numeric_features + categorical_features
train = pairs.loc[pairs["FLIGHT_DATE"] < cutoff].copy()
if train.empty:
    raise ValueError("Model development requires January-February training rows.")

## 1. Improve the first model using February validation

## 7. Improve the first model using February validation

The earlier Day 3 results showed a useful baseline, but we should not choose features or thresholds by repeatedly looking at March. This improvement round uses January for fitting and February for validation. It compares the original balanced logistic regression, a logistic model with domain-motivated feature engineering, and a Random Forest. Model and threshold selection use February F1; March is only an intermediate, previously inspected comparison and April is the fresh final holdout.

**Engineered features:** log transforms reduce the influence of the very long delay/slack tails; a short-turnaround flag and interaction represent the hypothesis that the same inbound delay is more consequential when little scheduled recovery time remains; sine/cosine hour features encode time-of-day cyclically. These are hypotheses to compare, not claims of causation.


In [8]:

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve

val_cutoff = pd.Timestamp("2026-02-01")
train_jan = pairs.loc[pairs["FLIGHT_DATE"] < val_cutoff].copy()
validation_feb = pairs.loc[(pairs["FLIGHT_DATE"] >= val_cutoff) & (pairs["FLIGHT_DATE"] < cutoff)].copy()
final_march = pairs.loc[(pairs["FLIGHT_DATE"] >= cutoff) & (pairs["FLIGHT_DATE"] < pd.Timestamp("2026-04-01"))].copy()
if train_jan.empty or validation_feb.empty or final_march.empty:
    raise ValueError("Improvement round requires January train, February validation, and March comparison rows.")

def add_engineered_features(frame):
    out = frame.copy()
    prev_delay = pd.to_numeric(out["PREV_ARR_DELAY_MIN"], errors="coerce").clip(lower=0)
    slack = pd.to_numeric(out["SCHEDULED_TURNAROUND_SLACK_MIN"], errors="coerce").clip(lower=0)
    hour = pd.to_numeric(out["DEPARTURE_HOUR_LOCAL"], errors="coerce")
    out["PREV_ARR_DELAY_LOG1P"] = np.log1p(prev_delay)
    out["SCHEDULED_SLACK_LOG1P"] = np.log1p(slack)
    out["SHORT_SLACK_LT_60"] = (slack < 60).astype("int8")
    out["PREV_DELAY_X_SHORT_SLACK"] = out["PREV_ARR_DELAY_LOG1P"] * out["SHORT_SLACK_LT_60"]
    out["DEPARTURE_HOUR_SIN"] = np.sin(2 * np.pi * hour / 24)
    out["DEPARTURE_HOUR_COS"] = np.cos(2 * np.pi * hour / 24)
    return out

engineered = add_engineered_features(pairs)
engineered_numeric = [
    "PREV_ARR_DELAY_LOG1P", "SCHEDULED_SLACK_LOG1P", "SHORT_SLACK_LT_60",
    "PREV_DELAY_X_SHORT_SLACK", "DEPARTURE_HOUR_SIN", "DEPARTURE_HOUR_COS",
]
engineered_categorical = ["ORIGIN", "DEST", "DAY_OF_WEEK"]
engineered_features = engineered_numeric + engineered_categorical

def make_preprocessor(numeric, categorical, scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric),
        ("categorical", categorical_pipeline, categorical),
    ], remainder="drop")

original_logistic = Pipeline([
    ("preprocess", make_preprocessor(numeric_features, categorical_features)),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=500, solver="liblinear")),
])
engineered_logistic = Pipeline([
    ("preprocess", make_preprocessor(engineered_numeric, engineered_categorical)),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=500, solver="liblinear")),
])
random_forest = Pipeline([
    ("preprocess", make_preprocessor(engineered_numeric, engineered_categorical, scale_numeric=False)),
    ("classifier", RandomForestClassifier(
        n_estimators=120, max_depth=14, min_samples_leaf=40,
        max_features="sqrt", class_weight="balanced_subsample",
        n_jobs=-1, random_state=42,
    )),
])
validation_models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Original balanced logistic": original_logistic,
    "Engineered balanced logistic": engineered_logistic,
    "Engineered random forest": random_forest,
}
X_train_jan = train_jan[feature_columns]
y_train_jan = train_jan[target]
X_valid_original = validation_feb[feature_columns]
y_valid = validation_feb[target]
X_train_jan_engineered = engineered.loc[train_jan.index, engineered_features]
X_valid_engineered = engineered.loc[validation_feb.index, engineered_features]

validation_rows = []
validation_thresholds = {}
for model_name, estimator in validation_models.items():
    is_engineered = model_name == "Engineered balanced logistic" or model_name == "Engineered random forest"
    x_fit = X_train_jan_engineered if is_engineered else X_train_jan
    x_valid = X_valid_engineered if is_engineered else X_valid_original
    estimator.fit(x_fit, y_train_jan)
    probabilities = estimator.predict_proba(x_valid)[:, 1]
    if model_name == "Majority baseline":
        chosen_threshold = 0.5
        predictions = estimator.predict(x_valid)
    else:
        precision_curve, recall_curve, thresholds = precision_recall_curve(y_valid, probabilities)
        f1_curve = 2 * precision_curve[:-1] * recall_curve[:-1] / np.maximum(precision_curve[:-1] + recall_curve[:-1], 1e-12)
        best_index = int(np.nanargmax(f1_curve))
        chosen_threshold = float(thresholds[best_index])
        predictions = (probabilities >= chosen_threshold).astype(int)
        validation_thresholds[model_name] = chosen_threshold
    validation_rows.append({
        "MODEL": model_name,
        "F1_AT_0_50": float(f1_score(y_valid, probabilities >= 0.5, zero_division=0)),
        "F1_AT_SELECTED_THRESHOLD": float(f1_score(y_valid, predictions, zero_division=0)),
        "SELECTED_THRESHOLD": float(chosen_threshold),
        "PRECISION": float(precision_score(y_valid, predictions, zero_division=0)),
        "RECALL": float(recall_score(y_valid, predictions, zero_division=0)),
        "ACCURACY": float(accuracy_score(y_valid, predictions)),
        "ROC_AUC": float(roc_auc_score(y_valid, probabilities)),
        "AVERAGE_PRECISION": float(average_precision_score(y_valid, probabilities)),
    })
validation_comparison = pd.DataFrame(validation_rows).sort_values("F1_AT_SELECTED_THRESHOLD", ascending=False)
validation_comparison.to_csv(REPORT_DIR / "day3_model_validation_comparison.csv", index=False)
print("January fit / February validation comparison (threshold selected on February for F1):")
print(validation_comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
selected_name = validation_comparison.loc[validation_comparison["MODEL"].ne("Majority baseline")].iloc[0]["MODEL"]
selected_threshold = validation_thresholds[selected_name]
print(f"Selected from February validation: {selected_name} at threshold {selected_threshold:.4f}")


January fit / February validation comparison (threshold selected on February for F1):
                       MODEL  F1_AT_0_50  F1_AT_SELECTED_THRESHOLD  SELECTED_THRESHOLD  PRECISION  RECALL  ACCURACY  ROC_AUC  AVERAGE_PRECISION
    Engineered random forest      0.6910                    0.6914              0.4988     0.6920  0.6908    0.8643   0.8656             0.7599
Engineered balanced logistic      0.6617                    0.6863              0.6861     0.7307  0.6470    0.8698   0.8620             0.7607
  Original balanced logistic      0.5859                    0.6142              0.5878     0.6423  0.5886    0.8373   0.8287             0.6221
           Majority baseline      0.0000                    0.0000              0.5000     0.0000  0.0000    0.7799   0.5000             0.2201
Selected from February validation: Engineered random forest at threshold 0.4988


## 2. Refit the validation winner for a March comparison

## 8. Refit the validation winner for a March comparison

After selection, refit that model on January and February together. Apply the threshold selected using February, then evaluate March as an intermediate comparison only. March has already been viewed during earlier baseline work; the new final holdout is April.


In [9]:

selected_estimator = validation_models[selected_name]
selected_engineered = selected_name in {"Engineered balanced logistic", "Engineered random forest"}
if selected_engineered:
    X_refit = engineered.loc[train.index, engineered_features]
    X_march = engineered.loc[final_march.index, engineered_features]
else:
    X_refit = train[feature_columns]
    X_march = final_march[feature_columns]
y_refit = train[target]
y_march = final_march[target]
selected_estimator.fit(X_refit, y_refit)
march_probability = selected_estimator.predict_proba(X_march)[:, 1]
march_prediction = (march_probability >= selected_threshold).astype(int)

improved_metrics = {
    "MODEL": selected_name + " (February-selected threshold)",
    "THRESHOLD": selected_threshold,
    "TEST_ROWS": int(len(y_march)),
    "TEST_POSITIVE_RATE": float(y_march.mean()),
    "ACCURACY": float(accuracy_score(y_march, march_prediction)),
    "PRECISION": float(precision_score(y_march, march_prediction, zero_division=0)),
    "RECALL": float(recall_score(y_march, march_prediction, zero_division=0)),
    "F1": float(f1_score(y_march, march_prediction, zero_division=0)),
    "ROC_AUC": float(roc_auc_score(y_march, march_probability)),
    "PR_AUC_AVERAGE_PRECISION": float(average_precision_score(y_march, march_probability)),
}
previous_test_results = pd.read_csv(REPORT_DIR / "day3_baseline_metrics.csv")
previous_test_results = previous_test_results.loc[previous_test_results["MODEL"].isin(["Majority baseline", "Balanced logistic regression"])].copy()
previous_test_results["THRESHOLD"] = 0.5
previous_test_results = previous_test_results.rename(columns={"PR_AUC_AVERAGE_PRECISION": "PR_AUC_AVERAGE_PRECISION"})
final_comparison = pd.concat([
    previous_test_results[["MODEL", "THRESHOLD", "TEST_ROWS", "TEST_POSITIVE_RATE", "ACCURACY", "PRECISION", "RECALL", "F1", "ROC_AUC", "PR_AUC_AVERAGE_PRECISION"]],
    pd.DataFrame([improved_metrics]),
], ignore_index=True)
final_comparison.to_csv(REPORT_DIR / "day3_model_improvement_march_metrics.csv", index=False)
selected_cm = confusion_matrix(y_march, march_prediction, labels=[0, 1])
selected_cm_table = pd.DataFrame(selected_cm, index=["Actual under 15 min", "Actual 15+ min"], columns=["Predicted under 15 min", "Predicted 15+ min"])
selected_cm_table.to_csv(REPORT_DIR / "day3_selected_model_march_confusion_matrix.csv")
print("March comparison; model and threshold selected using February only:")
print(final_comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("Selected-model confusion matrix:")
print(selected_cm_table.to_string())

improvement_summary = {
    "validation_period": "2026-02-01 through 2026-02-28",
    "initial_fit_period": "2026-01-01 through 2026-01-31",
    "refit_period": "2026-01-01 through 2026-02-28",
    "march_comparison_period": "2026-03-01 through 2026-03-31",
    "selection_metric": "February validation F1",
    "selected_model": selected_name,
    "selected_threshold": selected_threshold,
    "engineered_features": engineered_numeric,
    "categorical_features": engineered_categorical,
    "march_was_previously_viewed_in_baseline_exploration": True,
    "march_metrics": improved_metrics,
    "outputs": [
        "reports/metrics/day3_model_validation_comparison.csv",
        "reports/metrics/day3_model_improvement_march_metrics.csv",
        "reports/metrics/day3_selected_model_march_confusion_matrix.csv",
        "reports/metrics/day3_model_improvement_summary.json",
    ],
}
(REPORT_DIR / "day3_model_improvement_summary.json").write_text(json.dumps(improvement_summary, indent=2), encoding="utf-8")


March comparison; model and threshold selected using February only:
                                                 MODEL  THRESHOLD  TEST_ROWS  TEST_POSITIVE_RATE  ACCURACY  PRECISION  RECALL     F1  ROC_AUC  PR_AUC_AVERAGE_PRECISION
                                     Majority baseline     0.5000     114706              0.3022    0.6978     0.0000  0.0000 0.0000   0.5000                    0.3022
                          Balanced logistic regression     0.5000     114706              0.3022    0.7767     0.6071  0.7399 0.6670   0.8374                    0.7066
Engineered random forest (February-selected threshold)     0.4988     114706              0.3022    0.8479     0.7586  0.7285 0.7433   0.8777                    0.8272
Selected-model confusion matrix:
                     Predicted under 15 min  Predicted 15+ min
Actual under 15 min                   72004               8035
Actual 15+ min                         9412              25255


1376

### How to interpret model selection and the March comparison

### How to interpret this improvement round

The February validation table is the basis for choosing the model and threshold. March is an intermediate comparison; April is the fresh final holdout. Compare precision and recall alongside F1: lowering the threshold usually catches more delayed flights but also creates more false alerts. Select a threshold for the real operational objective rather than maximizing a score without context.


## 3. Fresh final evaluation on April

## 9. Fresh final evaluation on April

Candidate selection and threshold choice remain based on February validation. Now refit the selected Random Forest on all January-March records, then evaluate on April. April has not been used for feature/model/threshold selection. Do not make further changes based on April results; if you do, a later month will be needed for a fresh final test.


In [10]:

april_start = pd.Timestamp("2026-04-01")
train_through_march = pairs.loc[pairs["FLIGHT_DATE"] < april_start].copy()
april_test = pairs.loc[pairs["FLIGHT_DATE"] >= april_start].copy()
if train_through_march.empty or april_test.empty:
    raise ValueError("Fresh final evaluation requires January-March training data and April test data.")
y_final_train = train_through_march[target]
y_april = april_test[target]

# Refit all comparators on the same January-March history for a fair April comparison.
final_models = {
    "Majority baseline": (DummyClassifier(strategy="most_frequent"), False, 0.5),
    "Balanced logistic regression": (clone(original_logistic), False, 0.5),
    f"Selected {selected_name}": (clone(validation_models[selected_name]), selected_name in {"Engineered balanced logistic", "Engineered random forest"}, selected_threshold),
}
final_rows = []
selected_april_probability = None
selected_april_prediction = None
for model_name, (estimator, use_engineered, threshold) in final_models.items():
    if use_engineered:
        fit_x = engineered.loc[train_through_march.index, engineered_features]
        test_x = engineered.loc[april_test.index, engineered_features]
    else:
        fit_x = train_through_march[feature_columns]
        test_x = april_test[feature_columns]
    estimator.fit(fit_x, y_final_train)
    probability = estimator.predict_proba(test_x)[:, 1]
    prediction = (probability >= threshold).astype(int)
    final_rows.append({
        "MODEL": model_name,
        "THRESHOLD": threshold,
        "TRAIN_ROWS": int(len(y_final_train)),
        "APRIL_TEST_ROWS": int(len(y_april)),
        "APRIL_POSITIVE_RATE": float(y_april.mean()),
        "ACCURACY": float(accuracy_score(y_april, prediction)),
        "PRECISION": float(precision_score(y_april, prediction, zero_division=0)),
        "RECALL": float(recall_score(y_april, prediction, zero_division=0)),
        "F1": float(f1_score(y_april, prediction, zero_division=0)),
        "ROC_AUC": float(roc_auc_score(y_april, probability)),
        "PR_AUC_AVERAGE_PRECISION": float(average_precision_score(y_april, probability)),
    })
    if model_name.startswith("Selected "):
        selected_april_probability = probability
        selected_april_prediction = prediction

april_metrics = pd.DataFrame(final_rows)
april_metrics.to_csv(REPORT_DIR / "day3_april_final_metrics.csv", index=False)
april_cm = confusion_matrix(y_april, selected_april_prediction, labels=[0, 1])
april_cm_table = pd.DataFrame(april_cm, index=["Actual under 15 min", "Actual 15+ min"], columns=["Predicted under 15 min", "Predicted 15+ min"])
april_cm_table.to_csv(REPORT_DIR / "day3_april_final_confusion_matrix.csv")
print("April final holdout results; threshold/model selected using February validation:")
print(april_metrics.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("Selected model April confusion matrix:")
print(april_cm_table.to_string())

april_summary = {
    "data_file": str(PAIR_PATH.relative_to(ROOT)),
    "validation_model_selection": "January fit, February F1 validation",
    "final_training_period": "2026-01-01 through 2026-03-31",
    "fresh_test_period": "2026-04-01 through 2026-04-30",
    "selected_model": selected_name,
    "selected_threshold_from_february": selected_threshold,
    "train_rows": int(len(train_through_march)),
    "april_test_rows": int(len(april_test)),
    "april_positive_rate": float(y_april.mean()),
    "march_previously_inspected": True,
    "april_previously_used_for_selection": False,
    "metrics": april_metrics.to_dict(orient="records"),
    "outputs": [
        "reports/metrics/day3_april_final_metrics.csv",
        "reports/metrics/day3_april_final_confusion_matrix.csv",
        "reports/metrics/day3_april_final_summary.json",
    ],
}
(REPORT_DIR / "day3_april_final_summary.json").write_text(json.dumps(april_summary, indent=2), encoding="utf-8")


April final holdout results; threshold/model selected using February validation:
                            MODEL  THRESHOLD  TRAIN_ROWS  APRIL_TEST_ROWS  APRIL_POSITIVE_RATE  ACCURACY  PRECISION  RECALL     F1  ROC_AUC  PR_AUC_AVERAGE_PRECISION
                Majority baseline     0.5000      309576           114084               0.2558    0.7442     0.0000  0.0000 0.0000   0.5000                    0.2558
     Balanced logistic regression     0.5000      309576           114084               0.2558    0.7859     0.5611  0.7488 0.6415   0.8487                    0.6989
Selected Engineered random forest     0.4988      309576           114084               0.2558    0.8526     0.6948  0.7554 0.7239   0.8870                    0.8178
Selected model April confusion matrix:
                     Predicted under 15 min  Predicted 15+ min
Actual under 15 min                   75221               9683
Actual 15+ min                         7136              22044


1971

### Final evaluation rule

### Final evaluation rule

April is the fresh test month. Use this result as the headline generalization metric, alongside the confusion matrix and validation metrics. The logistic/majority comparators are refit on the same January-March data. The selected model keeps the threshold chosen on February. Avoid selecting or changing the model after comparing April results.
